In [20]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing import TypedDict, Annotated
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

In [21]:
LLM = ChatGroq(model="openai/gpt-oss-120b", temperature=0.5)

In [22]:
#search-tool 
search_tool = DuckDuckGoSearchRun(
    name = "search-tool",
    description=(
        "if the search related to updated knowleged from internet"
        "use this tool when user asks about current events"
        "new, current information, information require"
        "use internet for search"
    )
)

In [23]:
@tool
def search_tool(query: str)->str:
    """ search the internet for the query required from the user """
    
    decision = interrupt({
        "type": "approval",
        "tool": "search-tool",
        "query": query,
        "message": "The AI wants to search the internet",
        "instruction": "Approve this question (y/n)" 
    })
    
    if decision == 'reject':
        return {"message": [AIMessage(content="sorry cant search this for you!!")]}
    
    return search_tool.invoke(query)

In [24]:
tool = [search_tool]

In [25]:
llm_with_tool = LLM.bind_tools(tool)

In [26]:
class chatstate(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [27]:
def chat_node(state: chatstate):
    """ Chatbot with a search tool """
    messages = state['messages']
    
    response = llm_with_tool.invoke(messages)
    
    return {"messages": [response]}

tools = ToolNode(tool)

In [28]:
config = {"configurable": {"thread_id": "1234"}}

graph = StateGraph(chatstate)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tools)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")

checkpoint = InMemorySaver()

chatbot = graph.compile(checkpointer=checkpoint)


In [29]:
while True:
    user_input = input("type_here...")
    
    print("you: ", user_input)
    
    if user_input.lower().strip() in ["exit", "break", "thanks you"]:
        break
    
    # implementing hilt(first invoke)
    result = chatbot.invoke({
        "messages" : [HumanMessage(content=user_input)]}, config=config )
    
    # checking the intrrupt
    while "__interrupt__" in result:
        # if theres an intruppt then get the data
        interrupt_data = result["__interrupt__"][0]
        
        print("Human Loop Activated(approval_required)")
        # prinitng the search query
        print("search_query", interrupt_data.value["query"])
        
        # asking for the desisipn
        decision = input("Approve the query (y/n)")
        
        if decision.lower().strip() in ["yes", "y"]:
            result = chatbot.invoke(
                Command(resume="approve"),
                config=config
            )
        else:
            result = chatbot.invoke(
                Command(resume="reject"),
                config=config
            )
            
    print("AI: ", result["messages"][-1].content)
            

you:  hi
AI:  Hello! How can I assist you today?
you:  how are you
AI:  I'm doing great, thanks for asking! How can I help you today?
you:  can you tell me about ancient egyptions 
AI:  ### Ancient Egypt – A Quick Overview

| Aspect | Highlights |
|--------|-------------|
| **Geography** | Centered around the **Nile River** in what is now modern‑day Egypt. The river’s predictable flood cycle created fertile floodplains, while the surrounding deserts provided natural protection. |
| **Chronology** | **c. 3100 BC – 30 BC** (≈ 3,000 years) – divided into major periods: <br>• **Pre‑Dynastic** (c. 5500‑3100 BC) – village cultures, early writing. <br>• **Early Dynastic** (1st–2nd Dynasties, 3100‑2686 BC). <br>• **Old Kingdom** (3rd–6th Dynasties, 2686‑2181 BC) – “Age of the Pyramids.” <br>• **First Intermediate Period** (c. 2181‑2055 BC). <br>• **Middle Kingdom** (11th–14th Dynasties, 2055‑1650 BC). <br>• **Second Intermediate Period** (c. 1650‑1550 BC) – Hyksos rule. <br>• **New Kingdom** (